# Reset the Entity API database

Clears the **data** tables so you can re-run `test_api.ipynb` from a clean state:

- `entities`, `historical_entities`, `location`, `historical_location`, `validation_results`
- `responsible_contact`

The **rule-definition** tables (`val_rules`, `val_rule_conditions`, `val_checks`, `val_logic`) are left untouched, so you do **not** need to re-seed after a reset.

The API server can stay running while you do this (SQLite handles concurrent `DELETE`). Only the "delete the whole file" cell at the bottom needs the server stopped.

In [ ]:
import os
import sqlite3
import subprocess
import sys

import pandas as pd

# resolve the project root (folder containing run.py)
_here = os.getcwd()
while not os.path.exists(os.path.join(_here, "run.py")) and os.path.dirname(_here) != _here:
    _here = os.path.dirname(_here)
PROJECT_ROOT = _here

# same default the app uses (entity_api/config.py)
DB_NAME = os.environ.get("ENTITY_DB_NAME", "entity_database.db")
DB_PATH = os.path.join(PROJECT_ROOT, DB_NAME)

DATA_TABLES = ["entities", "historical_entities",
               "location", "historical_location",
               "validation_results", "latest_validation_rules",
               "validation_summary", "responsible_contact"]
RULE_TABLES = ["val_rules", "val_rule_conditions", "val_checks", "val_logic"]

print("project root:", PROJECT_ROOT)
print("database    :", DB_PATH, "| exists:", os.path.exists(DB_PATH))

In [ ]:
def table_counts(tables):
    rows = []
    with sqlite3.connect(DB_PATH) as conn:
        for t in tables:
            try:
                n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
            except sqlite3.OperationalError:
                n = None  # table not created yet
            rows.append({"table": t, "rows": n})
    return pd.DataFrame(rows)

print("DATA tables (cleared by reset):")
display(table_counts(DATA_TABLES))
print("RULE-definition tables (left untouched):")
display(table_counts(RULE_TABLES))

## Reset — clears data tables only

In [ ]:
def reset_data():
    if not os.path.exists(DB_PATH):
        print("no database file yet — nothing to reset.")
        return
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cleared = []
        for t in DATA_TABLES:
            try:
                cur.execute(f"DELETE FROM {t}")
                cleared.append(t)
            except sqlite3.OperationalError:
                pass  # table not created yet
        # reset AUTOINCREMENT so ids start again at 1
        try:
            cur.execute(
                "DELETE FROM sqlite_sequence WHERE name IN ({})".format(
                    ",".join("?" * len(DATA_TABLES))
                ),
                DATA_TABLES,
            )
        except sqlite3.OperationalError:
            pass  # no sqlite_sequence (no AUTOINCREMENT rows yet)
        conn.commit()
    print("cleared:", ", ".join(cleared) if cleared else "(nothing)")

reset_data()
print("\nafter reset:")
display(table_counts(DATA_TABLES))

## Optional — also wipe the rule-definition tables

Clears `val_rules`, `val_rule_conditions`, `val_checks`, `val_logic`.
After this the engine has **no rules**, so you must run the re-seed cell below
(or `python seed_validation_rules.py`) before the API can validate anything.

In [ ]:
def reset_rules():
    if not os.path.exists(DB_PATH):
        print("no database file yet — nothing to reset.")
        return
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cleared = []
        for t in RULE_TABLES:
            try:
                cur.execute(f"DELETE FROM {t}")
                cleared.append(t)
            except sqlite3.OperationalError:
                pass  # table not created yet
        try:
            cur.execute(
                "DELETE FROM sqlite_sequence WHERE name IN ({})".format(
                    ",".join("?" * len(RULE_TABLES))
                ),
                RULE_TABLES,
            )
        except sqlite3.OperationalError:
            pass
        conn.commit()
    print("cleared:", ", ".join(cleared) if cleared else "(nothing)")

reset_rules()
print("\nafter reset:")
display(table_counts(RULE_TABLES))

## Optional — re-seed the rule set

Only needed if the `val_*` tables are empty (fresh DB) or you edited `seed_validation_rules.py`.

In [ ]:
out = subprocess.run(
    [sys.executable, "seed_validation_rules.py"],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)
print(out.stdout or out.stderr)
display(table_counts(RULE_TABLES))

## Nuclear option — delete the whole database file

Stop the API server first (`--reload` keeps the file open on Windows). The API recreates an empty schema on next start; you then need the re-seed cell above.

In [ ]:
# import os
# if os.path.exists(DB_PATH):
#     os.remove(DB_PATH)
#     print("deleted", DB_PATH)
# else:
#     print("no db file")